<a href="https://colab.research.google.com/github/RMPlaysMCYT/Corn-Detection-System/blob/main/python/Notebooks/Corn_Seeds_Detection_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf

In [3]:
import keras

In [4]:
from keras._tf_keras.keras.preprocessing.image import ImageDataGenerator

In [5]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil
import cv2
import numpy as np
from pathlib import Path

base_path = '/content/drive/MyDrive/CornSeeds/NewCornData'
output_path = '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data'

# Create output folders
processed_folders = ['Healthy_Processed', 'Unhealthy_Processed']
for folder in processed_folders:
    os.makedirs(os.path.join(output_path, folder), exist_ok=True)

def zoom_and_resize_image(image_path, target_size=(256, 256), zoom_factor=1.2):
    """
    Zoom into the center of the image and resize to target size
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Get original dimensions
    h, w = img.shape[:2]

    # Calculate crop dimensions (zoom in)
    new_h = int(h / zoom_factor)
    new_w = int(w / zoom_factor)

    # Calculate center crop coordinates
    start_h = (h - new_h) // 2
    start_w = (w - new_w) // 2

    # Crop the center
    cropped = img[start_h:start_h + new_h, start_w:start_w + new_w]

    # Resize to target size
    resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LANCZOS4)

    return resized

def remove_background_and_zoom(image_path, target_size=(256, 256)):
    """
    Advanced: Remove background and focus on the corn seed
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply threshold to find the seed (adjust threshold as needed)
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Find the largest contour (assuming it's the seed)
        largest_contour = max(contours, key=cv2.contourArea)

        # Get bounding box
        x, y, w, h = cv2.boundingRect(largest_contour)

        # Add some padding
        padding = 20
        x = max(0, x - padding)
        y = max(0, y - padding)
        w = min(img.shape[1] - x, w + 2 * padding)
        h = min(img.shape[0] - y, h + 2 * padding)

        # Crop to the seed
        cropped = img[y:y+h, x:x+w]

        # Resize
        resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LANCZOS4)
        return resized

    # Fallback: use center zoom if no contour found
    return zoom_and_resize_image(image_path, target_size)

# Process all images
total_processed = 0

# Map source folders to output folders
folder_mapping = {
    'Healthy': {
        'users': ['Ronnel', 'Marjorie'],
        'output': 'Healthy_Processed'
    },
    'Unhealthy': {
        'users': ['Rodriguez', 'Ronnel', 'Marjorie'],
        'output': 'Unhealthy_Processed'
    }
}

for source_category, config in folder_mapping.items():
    for user in config['users']:
        source_path = os.path.join(base_path, source_category, user)
        output_folder = os.path.join(output_path, config['output'])

        if os.path.exists(source_path):
            # Get all image files
            image_files = []
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']:
                image_files.extend(Path(source_path).glob(ext))
                image_files.extend(Path(source_path).glob(ext.upper()))

            # Also check for files without extension (like IMG_20230724_123456)
            all_files = os.listdir(source_path)
            for f in all_files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                    if Path(os.path.join(source_path, f)) not in image_files:
                        image_files.append(Path(os.path.join(source_path, f)))

            print(f"Found {len(image_files)} images in {source_category}/{user}")

            for img_path in image_files:
                try:
                    # Option 1: Simple zoom and resize
                    # processed_img = zoom_and_resize_image(str(img_path), target_size=(256, 256), zoom_factor=1.3)

                    # Option 2: Advanced - remove background and zoom (recommended)
                    processed_img = remove_background_and_zoom(str(img_path), target_size=(256, 256))

                    if processed_img is not None:
                        # Generate output filename
                        filename = f"{source_category}_{user}_{img_path.stem}.jpg"
                        output_path_full = os.path.join(output_folder, filename)

                        # Save the processed image
                        cv2.imwrite(output_path_full, processed_img)
                        total_processed += 1

                        if total_processed % 10 == 0:
                            print(f"Processed {total_processed} images...")
                except Exception as e:
                    print(f"Error processing {img_path}: {e}")
        else:
            print(f"Warning: Path not found: {source_path}")

print(f"\n✅ Processing complete! Total images processed: {total_processed}")
print(f"📁 Output saved to: {output_path}")

# Verify the structure
print("\n📂 Folder structure created:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # Show subfolders only, not files
        for d in dirs:
            subindent = ' ' * 2 * (level + 1)
            print(f'{subindent}{d}/')

Found 464 images in Healthy/Ronnel
Processed 10 images...
Processed 20 images...
Processed 30 images...
Processed 40 images...
Processed 50 images...
Processed 60 images...
Processed 70 images...
Processed 80 images...
Processed 90 images...
Processed 100 images...
Processed 110 images...
Processed 120 images...
Processed 130 images...
Processed 140 images...
Processed 150 images...
Processed 160 images...
Processed 170 images...
Processed 180 images...
Processed 190 images...
Processed 200 images...
Processed 210 images...
Processed 220 images...
Processed 230 images...
Processed 240 images...
Processed 250 images...
Processed 260 images...
Processed 270 images...
Processed 280 images...
Processed 290 images...
Processed 300 images...
Processed 310 images...
Processed 320 images...
Processed 330 images...
Processed 340 images...
Processed 350 images...
Processed 360 images...
Processed 370 images...
Processed 380 images...
Processed 390 images...
Processed 400 images...
Processed 410 

In [7]:
import os
print(os.listdir('/content/drive/MyDrive/CornSeeds/NewCornData/processed_data'))

['Healthy_Processed', 'Unhealthy_Processed']


In [8]:
training_set1 = datagen.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

Found 1985 images belonging to 2 classes.


In [9]:
training_datagenerator1 = ImageDataGenerator(rescale = 1./255)

In [10]:
testing_set1 = training_datagenerator1.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

Found 1985 images belonging to 2 classes.


In [11]:
cnn_process1 = tf.keras.models.Sequential()

# FIRST LAYER

In [12]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=[64, 64, 3]
    )
)

cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# SECOND LAYER

In [13]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
    )
)
cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

In [14]:
cnn_process1.add(tf.keras.layers.Flatten())

In [15]:
cnn_process1.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn_process1.add(tf.keras.layers.Dropout(0.5))

In [16]:
cnn_process1.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [17]:
cnn_process1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=50)

Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 479s 8s/step - accuracy: 0.6327 - loss: 0.6280 - val_accuracy: 0.7652 - val_loss: 0.5151
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 223ms/step - accuracy: 0.7275 - loss: 0.5066 - val_accuracy: 0.7652 - val_loss: 0.4462
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 227ms/step - accuracy: 0.7526 - loss: 0.4634 - val_accuracy: 0.7708 - val_loss: 0.4227
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.7647 - loss: 0.4428 - val_accuracy: 0.7708 - val_loss: 0.4220
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 230ms/step - accuracy: 0.7678 - loss: 0.4196 - val_accuracy: 0.7723 - val_loss: 0.3945
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 227ms/step - accuracy: 0.7592 - loss: 0.4170 - val_accuracy: 0.7718 - val_loss: 0.3960
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.7673 - loss: 0.3998 - val_accuracy: 0.7723 - val_loss: 0.3819
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.7673 - loss: 0.3906 - val_accura

* EPOCH 200

In [ ]:
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=200)

Epoch 1/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 259ms/step - accuracy: 0.7510 - loss: 0.4533 - val_accuracy: 0.7590 - val_loss: 0.4367
Epoch 2/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 261ms/step - accuracy: 0.7535 - loss: 0.4481 - val_accuracy: 0.7565 - val_loss: 0.4477
Epoch 3/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 259ms/step - accuracy: 0.7560 - loss: 0.4433 - val_accuracy: 0.7735 - val_loss: 0.4332
Epoch 4/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 258ms/step - accuracy: 0.7550 - loss: 0.4536 - val_accuracy: 0.7595 - val_loss: 0.4558
Epoch 5/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 249ms/step - accuracy: 0.7645 - loss: 0.4467 - val_accuracy: 0.7595 - val_loss: 0.4394
Epoch 6/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 276ms/step - accuracy: 0.7530 - loss: 0.4551 - val_accuracy: 0.7605 - val_loss: 0.4375
Epoch 7/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 255ms/step - accuracy: 0.7530 - loss: 0.4470 - val_accuracy: 0.7640 - val_loss: 0.4316
Epoch 8/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 259ms/step - accuracy: 0.7625 - loss: 0.4476 - 

* EPOCH 300

In [ ]:
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=300)

Epoch 1/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 299ms/step - accuracy: 0.6685 - loss: 0.5839 - val_accuracy: 0.7110 - val_loss: 0.5278
Epoch 2/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 258ms/step - accuracy: 0.7175 - loss: 0.5268 - val_accuracy: 0.7175 - val_loss: 0.5160
Epoch 3/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 255ms/step - accuracy: 0.7205 - loss: 0.5069 - val_accuracy: 0.7280 - val_loss: 0.4883
Epoch 4/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 254ms/step - accuracy: 0.7275 - loss: 0.5185 - val_accuracy: 0.7280 - val_loss: 0.4900
Epoch 5/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 258ms/step - accuracy: 0.7235 - loss: 0.5061 - val_accuracy: 0.7270 - val_loss: 0.4855
Epoch 6/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 255ms/step - accuracy: 0.7210 - loss: 0.4978 - val_accuracy: 0.7330 - val_loss: 0.4736
Epoch 7/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 256ms/step - accuracy: 0.7285 - loss: 0.4893 - val_accuracy: 0.7330 - val_loss: 0.4709
Epoch 8/300
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - accuracy: 0.7305 - loss: 0.5093 - 

In [ ]:
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=350)

Epoch 1/350
63/63 ━━━━━━━━━━━━━━━━━━━━ 25s 315ms/step - accuracy: 0.6365 - loss: 0.6103 - val_accuracy: 0.7260 - val_loss: 0.5117
Epoch 2/350
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 361ms/step - accuracy: 0.7150 - loss: 0.5270 - val_accuracy: 0.7265 - val_loss: 0.5274
Epoch 3/350
42/63 ━━━━━━━━━━━━━━━━━━━━ 7s 368ms/step - accuracy: 0.7027 - loss: 0.5184

In [19]:
cnn_process1.save('/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_2.keras')

In [ ]:
import tensorflow as tf

# Path to your saved model
model_path = '/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_1.keras'

# Load the model
print(f"Loading model from {model_path}...")
loaded_cnn_model = tf.keras.models.load_model(model_path)
print("Model loaded successfully!")

In [ ]:
import numpy as np
from keras.preprocessing import image
import os

# Directory for testing images
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Setup generator for testing
prediction_datagen = image.ImageDataGenerator(rescale=1./255)
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

# Run prediction using the LOADED model
predictions = loaded_cnn_model.predict(prediction_generator)

# Mapping and output
idx_to_class = {0: 'healthy', 1: 'unhealthy'}
filenames = prediction_generator.filenames

print("\n--- Predictions using loaded model ---")
for i, score in enumerate(predictions):
    predicted_class = idx_to_class[1 if score[0] > 0.5 else 0]
    image_name = os.path.basename(filenames[i])
    print(f"Image: {image_name}, Predicted: {predicted_class} (Confidence: {score[0]:.4f})")

* VERSION 0.0.2 KERAS

* VERSION 0.1.0 KERAS

In [20]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_process1)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_2.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpi89xmzke'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137070975808336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070975809488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070975808912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070975810256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070975808144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070975810832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070935689360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137070935690128: TensorSpec(shape=(), dtype=tf.resource, name=None)


* VERSION 0.0.2 TFLITE

* VERSION 0.1.0 TFLITE

* PROCESSING

In [ ]:
import tensorflow as tf

# Path to your saved model
model_path = '/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_1.keras'

# Load the model
print(f"Loading model from {model_path}...")
cnn_process1 = tf.keras.models.load_model(model_path)

print("Model loaded successfully! You can now use 'cnn_process1' for predictions.")

Loading model from /content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_1.keras...
Model loaded successfully! You can now use 'cnn_process1' for predictions.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

Found 3007 images belonging to 3 classes.

--- Starting predictions for 3007 files ---
Image: aug_3_2036712710_WangDataa35.jpg, Predicted: healthy (Confidence: 0.0000)
Image: aug_3_203714993_WangDataa46.jpg, Predicted: healthy (Confidence: 0.0002)
Image: aug_3_203818312_WangDataa60.jpg, Predicted: healthy (Confidence: 0.0000)
Image: aug_3_2040426164_WangDataa43.jpg, Predicted: healthy (Confidence: 0.0000)
Image: aug_3_2040816385_WangDataa36.jpg, Predicted: healthy (Confidence: 0.0001)
Image: aug_3_2042297005_WangDataa41.jpg, Predicted: healthy (Confidence: 0.0000)
Skipping file healthy/aug_3_2043542508_WangDataa64.jpg due to error: cannot identify image file <_io.BytesIO object at 0x7a38f3f8df30>
Image: healthy_1.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_10.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_100.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_1000.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_101.jpg, Predicted: healt

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

# Counters for statistics
healthy_count = 0
unhealthy_count = 0
processed_count = 0

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        # Update counters
        if predicted_class_idx == 0:
            healthy_count += 1
        else:
            unhealthy_count += 1
        processed_count += 1

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

# Display summary statistics
if processed_count > 0:
    healthy_percent = (healthy_count / processed_count) * 100
    unhealthy_percent = (unhealthy_count / processed_count) * 100

    print("\n" + "="*30)
    print("PREDICTION SUMMARY")
    print("="*30)
    print(f"Total seeds processed: {processed_count}")
    print(f"Healthy seeds:   {healthy_count} ({healthy_percent:.2f}%)")
    print(f"Unhealthy seeds: {unhealthy_count} ({unhealthy_percent:.2f}%)")
    print("="*30)
else:
    print("\nNo images were successfully processed.")

Found 3007 images belonging to 3 classes.

--- Starting predictions for 3007 files ---
Image: aug_3_2036712710_WangDataa35.jpg, Predicted: healthy (Confidence: 0.0000)
Image: aug_3_203714993_WangDataa46.jpg, Predicted: healthy (Confidence: 0.0002)
Image: aug_3_203818312_WangDataa60.jpg, Predicted: healthy (Confidence: 0.0000)
Image: aug_3_2040426164_WangDataa43.jpg, Predicted: healthy (Confidence: 0.0000)
Image: aug_3_2040816385_WangDataa36.jpg, Predicted: healthy (Confidence: 0.0001)
Image: aug_3_2042297005_WangDataa41.jpg, Predicted: healthy (Confidence: 0.0000)
Skipping file healthy/aug_3_2043542508_WangDataa64.jpg due to error: cannot identify image file <_io.BytesIO object at 0x7a3882ee55d0>
Image: healthy_1.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_10.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_100.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_1000.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_101.jpg, Predicted: healt

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

# Counters for statistics
healthy_count = 0
unhealthy_count = 0
processed_count = 0

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        # Update counters
        if predicted_class_idx == 0:
            healthy_count += 1
        else:
            unhealthy_count += 1
        processed_count += 1

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

# Display summary statistics
if processed_count > 0:
    healthy_percent = (healthy_count / processed_count) * 100
    unhealthy_percent = (unhealthy_count / processed_count) * 100

    print("\n" + "="*30)
    print("PREDICTION SUMMARY")
    print("="*30)
    print(f"Total seeds processed: {processed_count}")
    print(f"Healthy seeds:   {healthy_count} ({healthy_percent:.2f}%)")
    print(f"Unhealthy seeds: {unhealthy_count} ({unhealthy_percent:.2f}%)")
    print("="*30)
else:
    print("\nNo images were successfully processed.")

Found 1985 images belonging to 2 classes.

--- Starting predictions for 1985 files ---
Image: Healthy_Marjorie_IMG_7522.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7523.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7524.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7525.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7526.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7527.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7528.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7529.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7530.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7531.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7532.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7533.jpg, Predicted: health

### CNN Version 2: Optimized for 256x256 resolution
Since the preprocessing output is 256x256, we should update the model architecture to utilize this resolution instead of downsampling to 64x64.

In [11]:
cnn_v2 = tf.keras.models.Sequential([
    # Input Layer
    tf.keras.layers.Input(shape=(256, 256, 3)),

    # First Convolutional Block
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

    # Second Convolutional Block
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

    # Third Convolutional Block (Capturing deeper features)
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

    # Flatten and Dense Layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

cnn_v2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_v2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,840,001 (56.61 MB)

 Trainable params: 14,839,553 (56.61 MB)

 Non-trainable params: 448 (1.75 KB)

### Updated Generators for 256x256
We also need to update the `target_size` in the data generators to match the new input shape.

In [12]:
train_generator_v2 = datagen.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary'
)

test_generator_v2 = training_datagenerator1.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary'
)

Found 1985 images belonging to 2 classes.
Found 1985 images belonging to 2 classes.
